# W&B sweep config vs Python API for hyperparameter optimization

This notebook compares two ways to define hyperparameter sweeps in Weights & Biases: **declarative YAML config files** and **programmatic Python dicts** passed to `wandb.sweep()`. Both approaches target the same RandomForest classifier on the Iris dataset so the comparison is about ergonomics, not model quality.

The question is straightforward: when should you reach for a `.yaml` file and when does it make more sense to keep the config inline in Python?

## Setup

Install wandb if not already available and import dependencies. The notebook assumes you have a W&B account and are logged in (`wandb login`).

In [ ]:
import wandb
import yaml
from pathlib import Path
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings

warnings.filterwarnings("ignore")
print(f"wandb version: {wandb.__version__}")

Load the Iris dataset — simple enough that we can focus on the sweep mechanics.

In [ ]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train: {len(X_train)} samples | Test: {len(X_test)} samples")

---
## Approach 1 — YAML sweep config file

The declarative approach: write a `sweep_config.yaml` file with the search space, method, and metric, then load it with `yaml.safe_load()` and pass to `wandb.sweep()`. This keeps the configuration separate from the training code — useful for reuse across teams or when the sweep config is maintained by someone who doesn't touch the Python code.

In [ ]:
SWEEP_YAML = """
program: train.py
method: bayes
metric:
  name: accuracy
  goal: maximize
parameters:
  n_estimators:
    values: [50, 100, 200]
  max_depth:
    values: [3, 5, 7, null]
  min_samples_split:
    min: 2
    max: 10
  min_samples_leaf:
    min: 1
    max: 5
early_terminate:
  type: hyperband
  min_iter: 3
"""

yaml_path = Path("/tmp/sweep_config.yaml")
yaml_path.write_text(SWEEP_YAML)

with open(yaml_path) as f:
    yaml_config = yaml.safe_load(f)

print("YAML config loaded:")
print(yaml.dump(yaml_config, default_flow_style=False).strip())

Now register the sweep with `wandb.sweep()` using the YAML-derived config. This returns a sweep ID that the agent can use.

In [ ]:
project_name = "sweep-compare-demo"

def train_yaml() -> None:
    with wandb.init() as run:
        cfg = run.config
        model = RandomForestClassifier(
            n_estimators=cfg.get("n_estimators", 100),
            max_depth=cfg.get("max_depth"),
            min_samples_split=cfg.get("min_samples_split", 2),
            min_samples_leaf=cfg.get("min_samples_leaf", 1),
            random_state=42,
        )
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        wandb.log({"accuracy": acc})

yaml_sweep_id = wandb.sweep(yaml_config, project=project_name)
print(f"YAML-based sweep registered: {yaml_sweep_id}")

---
## Approach 2 — Python API (programmatic config)

The programmatic approach defines the sweep config as a plain Python dict and passes it directly to `wandb.sweep()`. No file I/O, no YAML parsing — the config lives in the same module as the training function. This is convenient during exploration when the sweep boundaries are evolving rapidly.

In [ ]:
python_config = {
    "method": "bayes",
    "metric": {"name": "accuracy", "goal": "maximize"},
    "parameters": {
        "n_estimators": {"values": [50, 100, 200]},
        "max_depth": {"values": [3, 5, 7, None]},
        "min_samples_split": {"min": 2, "max": 10},
        "min_samples_leaf": {"min": 1, "max": 5},
    },
    "early_terminate": {"type": "hyperband", "min_iter": 3},
}

def train_python() -> None:
    with wandb.init() as run:
        cfg = run.config
        model = RandomForestClassifier(
            n_estimators=cfg.get("n_estimators", 100),
            max_depth=cfg.get("max_depth"),
            min_samples_split=cfg.get("min_samples_split", 2),
            min_samples_leaf=cfg.get("min_samples_leaf", 1),
            random_state=42,
        )
        model.fit(X_train, y_train)
        acc = accuracy_score(y_test, model.predict(X_test))
        wandb.log({"accuracy": acc})

python_sweep_id = wandb.sweep(python_config, project=project_name)
print(f"Python API sweep registered: {python_sweep_id}")

---
## Comparison

Both approaches produced a sweep ID and registered the same search space. The difference is entirely in how the config is authored and maintained.

| Aspect | YAML config | Python dict |
|--------|-------------|-------------|
| **Portability** | Drop the `.yaml` file into any project — no code changes needed | Lives inside the Python module; requires a code change to reuse |
| **Editing** | Any text editor or YAML-aware IDE | Requires Python knowledge to avoid syntax errors |
| **Dynamic configs** | Harder — would need a template engine or codegen | Trivial — build the dict programmatically (e.g. derive ranges from data) |
| **Validation** | YAML parser catches structure errors only | Same — dict keys are not schema-validated by wandb.sweep() |
| **Version control diff** | Clean — changes to sweep params are just YAML line diffs | Also clean — the dict is right there in the source |
| **Team workflow fit** | Good when MLE or PM defines sweeps without touching training code | Good when the same person writing training code also owns the sweep |

One thing I'm not fully settled on: the YAML config references a `program: train.py` field that `wandb.sweep()` uses when launching agents from the CLI. When calling `wandb.sweep()` from Python this field is ignored — but it can be confusing if you copy a YAML config that expects a CLI entry point. The wandb docs suggest `program` is mandatory for CLI sweeps but irrelevant for SDK-based launches.

## Verify — check that both sweeps were created

Query the W&B API to confirm both sweeps exist and have the expected configuration.

In [ ]:
api = wandb.Api()

for label, sid in [("YAML", yaml_sweep_id), ("Python", python_sweep_id)]:
    sweep = api.sweep(f"my-entity/{project_name}/{sid}")
    print(f"\n{'='*50}")
    print(f"  {label}  |  sweep_id: {sid}")
    print(f"{'='*50}")
    print(f"  Method:         {sweep.config.get('method')}")
    print(f"  Metric:         {sweep.config.get('metric')}")
    print(f"  Parameters:     {list(sweep.config.get('parameters', {}).keys())}")
    print(f"  Early terminate: {sweep.config.get('early_terminate', {}).get('type', 'none')}")
    print(f"  Run count:      {len(sweep.runs)}")

## Summary

- The **YAML approach** shines when the sweep config is a team artifact — separate from code, editable by non-Python users, and portable across projects.
- The **Python dict approach** is quicker to iterate on during development. You can compute parameter ranges dynamically, omit fields conditionally, and keep everything in one file.
- Both produce the same sweep on the W&B backend. The choice is about workflow, not capability.
- If you're not sure, start with the Python dict during exploration and extract to YAML once the sweep stabilises — that way you avoid context-switching to a separate file while you're still figuring out the search space.